 # Assignment 7 — Validation Accuracy + Model Evaluation

## Dataset

| Hours	| Sleep	| Passed |
| :---: | :---: | :---: |
| 1	| 2	| 0 |
| 2	| 1	| 0 |
| 3	| 1	| 0 |
| 4	| 5	| 1 |
| 5	| 4	| 1 |
| 6	| 5	| 1 |
| 2	| 3	| 0 |
| 5	| 2	| 1 |

- `Features`: (hours, sleep)

- `Target`: 0 or 1

_ _ _

## Part A — Train/Test Split

Manually split:

- Training: first 6 samples

- Testing: last 2 samples

In [22]:
import torch

x = torch.tensor([
    [1,2],
    [2,1],
    [3,1],
    [4,5],
    [5,4],
    [6,5],
    [2,3],
    [5,2]
], dtype=torch.float32)

y = torch.tensor([0,0,0,1,1,1,0,1])

x_train=x[:6]
y_train=y[:6]

x_test=x[6:]
y_test=y[6:]

print(x_train,y_train)
print(x_test,y_test)

tensor([[1., 2.],
        [2., 1.],
        [3., 1.],
        [4., 5.],
        [5., 4.],
        [6., 5.]]) tensor([0, 0, 0, 1, 1, 1])
tensor([[2., 3.],
        [5., 2.]]) tensor([0, 1])


## Part B — Create Dataset + DataLoader

- Create : Train dataset, Train dataloader

- Batch size: 2

In [23]:
import torch
from torch.utils.data import Dataset,DataLoader

# DataSet
class TrainDataset(Dataset):

    def __init__(self):
        self.features=x_train
        self.labels=y_train
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):
        x=self.features[idx]
        y=self.labels[idx]
        return x,y
    
dataset=TrainDataset()

# DataLoader

loader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

## Part C — Build Model

Architecture:

- Linear(2,16)
- ReLU
- Linear(16,2)

In [24]:
import torch
import torch.nn as nn

class ClassifierModel(nn.Module):

    def __init__(self):

        super().__init__()
        self.linear_1=nn.Linear(2,16)
        self.relu=nn.ReLU()
        self.linear_2=nn.Linear(16,2)

    def forward(self,x):

        x=self.linear_1(x)
        x=self.relu(x)
        x=self.linear_2(x)

        return x
    
model = ClassifierModel()



## Part D — Training

- Train for : 300 epochs

- Print loss every 50 epochs.

- Use : CrossEntropyLoss, Adam

In [25]:
import torch.nn as nn
import torch.optim as optim

loss_fn=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.01)

model.train()

for epoch in range(300):
    epoch_loss=0
    for batch_x,batch_y in loader:

        predictions=model(batch_x)
        loss=loss_fn(predictions,batch_y)
        epoch_loss+=loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 50==0:
        print(f"Epoch {epoch}:", epoch_loss)

Epoch 0: 5.193475753068924
Epoch 50: 0.14664537087082863
Epoch 100: 0.03089633584022522
Epoch 150: 0.013378336792811751
Epoch 200: 0.007395727909170091
Epoch 250: 0.004716944415122271


## Part E — Accuracy Calculation

Evaluate on test set.

Steps:

- 1.disable gradients
- 2.get logits
- 3.argmax predictions
- 4.compare with true labels
- 5.compute accuracy manually

In [28]:
model.eval()

with torch.no_grad():

    logits=model(x_test)

    preds=torch.argmax(logits,dim=1)

    correct=(preds == y_test).sum().item()

    total=len(y_test)

    accuracy=(correct / total) * 100

print("Logits:\n", logits)
print("Predictions:", preds)
print("True Labels:", y_test)
print(f"Accuracy: {accuracy:.2f}%")

Logits:
 tensor([[1.3489, 0.0944],
        [1.5820, 1.4026]])
Predictions: tensor([0, 0])
True Labels: tensor([0, 1])
Accuracy: 50.00%
